In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalog", "dbr_dev", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

#small agregation tables
GOLD_LIVE_KPI = f"{CATALOG}.{SCHEMA}.gold_live_kpi"
GOLD_ROUTE_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_route_summary"
GOLD_DELAY_DISTRIBUTION = f"{CATALOG}.{SCHEMA}.gold_delay_distribution"
GOLD_DESTINATION_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_destination_summary"
GOLD_FLEET_CURRENT = f"{CATALOG}.{SCHEMA}.gold_fleet_current"
GOLD_PERIOD_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_period_summary"

DIM_VEHICLE = f"{CATALOG}.{SCHEMA}.dim_vehicle"
DIM_ROUTE = f"{CATALOG}.{SCHEMA}.dim_route"
DIM_DESTINATION = f"{CATALOG}.{SCHEMA}.dim_destination"
DIM_TIME = f"{CATALOG}.{SCHEMA}.dim_time"

FACT_VEHICLE_STATUS = f"{CATALOG}.{SCHEMA}.fact_vehicle_status"

silver = spark.read.table(SILVER)
fact_df = spark.read.table(FACT_VEHICLE_STATUS)
dim_vehicle = spark.read.table(DIM_VEHICLE)
dim_route = spark.read.table(DIM_ROUTE)
dim_destination = spark.read.table(DIM_DESTINATION)
dim_time = spark.read.table(DIM_TIME)


In [0]:
gold_period_summary = (
    fact_df.agg(
        F.countDistinct("vehicle_key").alias("total_vehicles"),        # fleet seen over the period
        F.count("*").alias("total_readings"),
        F.round(F.avg(F.when(F.col("has_trip"), F.col("delay_min"))), 2).alias("avg_delay_min"),
        F.round(F.max(F.when(F.col("has_trip"), F.col("delay_min"))), 2).alias("max_delay_min"),
        F.round(F.avg(F.col("is_delayed").cast("int")) * 100, 1).alias("delayed_reading_pct"),  # % of readings delayed
        F.round(F.avg(F.when(F.col("is_moving"), F.col("speed"))), 2).alias("avg_speed"))
)

In [0]:
(
    gold_period_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_PERIOD_SUMMARY)
)

In [0]:
data = (
    fact_df
    .join(dim_route.select("route_key", "routeId", "routeShortName"), on="route_key", how="left")
    .join(dim_vehicle.select("vehicle_key", "transportationType"), on="vehicle_key", how="left")
)

gold_route_summary = (
    data.groupBy("routeId", "routeShortName", "transportationType")
    .agg(
        F.countDistinct("vehicle_key").alias("active_vehicles"),
        F.round(
            F.avg(F.when(F.col("has_trip"),F.col("delay_min"))),2
        ).alias("avg_delay_min"),

        F.round(
            F.max(F.when(F.col("has_trip"),F.col("delay_min"))),2
        ).alias("max_delay_min"),

        F.round(
            F.avg(F.when( F.col("is_moving"), F.col("speed"))),2
        ).alias("avg_speed"),

        F.countDistinct(
            F.when( F.col("is_delayed"), F.col("vehicleId"))
        ).alias("delayed_vehicles"),

        F.countDistinct(
            F.when(F.col("is_stopped"),F.col("vehicleId") )
        ).alias("stopped_vehicles")
    )
)
 

In [0]:
(
    gold_route_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ROUTE_SUMMARY)
)

In [0]:
#delay distribution
data = (
    fact_df.join(dim_vehicle.select("vehicle_key", "transportationType"), on="vehicle_key", how="left")
)
gold_delay_distribution = (
    data
    .filter(F.col("has_trip"))
    .groupBy("delay_bucket", "transportationType")
    .agg(
        F.count("*").alias("records_count"),
        F.countDistinct("vehicle_key").alias("vehicles_count")
    )
)

In [0]:

(
    gold_delay_distribution.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DELAY_DISTRIBUTION)
)

In [0]:
data = (
    fact_df
    .join(dim_route.select("route_key", "routeShortName"), on="route_key", how="left")
    .join(dim_destination.select("destination_key", "headsign"), on="destination_key", how="left")
    .join(dim_vehicle.select("vehicle_key", "transportationType"), on="vehicle_key", how="left")
)

gold_destination_summary = (
data.filter(
        F.col("has_trip") & F.col("headsign").isNotNull()
    )
    .groupBy(
        "routeShortName", # which line
        "headsign", # destination / direction
        "transportationType"
    )
    .agg(
        F.countDistinct("vehicle_key").alias("active_vehicles"),
        F.round(F.avg("delay_min"),2).alias("avg_delay_min"),
        F.round(F.avg(F.when(F.col("is_moving"),F.col("speed"))),2).alias("avg_speed"),
        F.countDistinct(F.when( F.col("is_delayed"),F.col("vehicle_key"))).alias("delayed_vehicles")
    )
)

In [0]:
(
    gold_destination_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DESTINATION_SUMMARY)
)

In [0]:

# latest reading per vehicle -> one point per vehicle for the MAP
w = Window.partitionBy("vehicle_key").orderBy(F.col("event_time_local").desc())
current = (
    fact_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1).drop("rn")
)

data = (
    current
    .join(dim_route.select("route_key", "routeShortName"), on="route_key", how="left")
    .join(dim_destination.select("destination_key", "headsign"), on="destination_key", how="left")
    .join(dim_vehicle.select("vehicle_key", "vehicleId", "vehicleCode" ,"transportationType", "model"), on="vehicle_key", how="left")
)

gold_fleet_current = (
    data.select("vehicleId","vehicleCode","routeShortName","headsign",
        "lat","lon","delay","delay_min","delay_bucket","speed","is_moving",
        "transportationType","model", "event_time_local")    
)

In [0]:
(gold_fleet_current.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_FLEET_CURRENT))

In [0]:
fc = spark.read.table(GOLD_FLEET_CURRENT)
gold_life_kpi = (
    fc.agg(
        F.max("event_time_local").alias("snapshot_time"),
        F.countDistinct("vehicleId").alias("active_vehicles"),
        F.countDistinct(F.when(F.col("is_moving")==False, F.col("vehicleId"))).alias("stopped_vehicles"),
        F.countDistinct(F.when(F.col("delay")>120, F.col("vehicleId"))).alias("delayed_vehicles"),
        F.round(F.avg("delay_min"),2).alias("avg_delay_min"),
        F.round(F.max("delay_min"),2).alias("max_delay_min")
    )
)

In [0]:
(gold_live_kpi.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_LIVE_KPI))